<a href="https://colab.research.google.com/github/Halidh-Ahamed/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1

One finding from the FlyRank paper is that machine learning models can help identify content that is likely to decline in performance so it can be refreshed earlier.

- **Where does the label come from?**
  The label is created from historical search-performance data, where past content performance is used to determine whether a page declined.

- **Does the validation design carry the claim?**
  The claim is stronger if the model is evaluated on unseen pages or future data rather than a simple random split. This helps show that the model generalizes instead of memorizing patterns.

---

### Finding 2

Another finding is that using multiple search-related features improves prediction compared to relying on a single metric.

- **Where does the label come from?**
  The label is derived from observed historical outcomes in the search-performance data rather than being manually assigned.

- **Does the validation design carry the claim?**
  The claim is supported only if the evaluation avoids data leakage and uses an honest validation strategy, such as grouped or time-aware splits. Otherwise, the reported performance may be overly optimistic.

---

### My methodology questions

1. How exactly is the decline label defined, and what threshold is used?
2. Would the model perform similarly on completely new clients or on future time periods?
3. Which features contribute the most to the model's predictions, and are any of them at risk of causing data leakage?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [5]:
# Load March 2026 data


HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

feature_df = con.sql(f"""
SELECT
    report_date,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position,
    client_has_gsc,
    client_has_ga4,
    gsc_data_available
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2026-03'
""").df()

print(feature_df.shape)
feature_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(9841378, 8)


,report_date,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,client_has_gsc,client_has_ga4,gsc_data_available
0,2026-03-01,content_b7e512995f79d5a6,20,0,67,True,False,True
1,2026-03-01,content_05597932fe4da067,1,0,0,True,False,True
2,2026-03-01,content_7a105f548d9c6916,125,1,616,True,False,True
3,2026-03-01,content_905aa32a0230694e,7,0,28,True,False,True
4,2026-03-01,content_a3ea9792f793ec72,11,0,25,True,False,True


In [6]:
# ==========================
# Feature engineering
# ==========================

feature_df["avg_position"] = (
    feature_df["gsc_sum_position"] /
    feature_df["gsc_impressions"].replace(0, 1)
)

feature_df["ctr"] = (
    feature_df["gsc_clicks"] /
    feature_df["gsc_impressions"].replace(0, 1)
)

# Create a binary target:
# 1 = page received at least one click
# 0 = page received no clicks
feature_df["target"] = (feature_df["gsc_clicks"] > 0).astype(int)

# Keep only the columns we'll model with
model_df = feature_df[
    [
        "content_hash_id",
        "gsc_impressions",
        "avg_position",
        "ctr",
        "client_has_gsc",
        "client_has_ga4",
        "gsc_data_available",
        "target",
    ]
].dropna()

print(model_df.shape)
model_df.head()

(9841378, 8)


,content_hash_id,gsc_impressions,avg_position,ctr,client_has_gsc,client_has_ga4,gsc_data_available,target
0,content_b7e512995f79d5a6,20,3.350000,0.000,True,False,True,0
1,content_05597932fe4da067,1,0.000000,0.000,True,False,True,0
2,content_7a105f548d9c6916,125,4.928000,0.008,True,False,True,1
3,content_905aa32a0230694e,7,4.000000,0.000,True,False,True,0
4,content_a3ea9792f793ec72,11,2.272727,0.000,True,False,True,0


In [9]:
# ==========================
# BEFORE: Ordinary train/test split
# ==========================

X = model_df.drop(columns=["target", "content_hash_id", "ctr"])
y = model_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf = RandomForestClassifier(
    n_estimators=30,
    max_depth=10,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("=== BEFORE (Ordinary Split) ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

=== BEFORE (Ordinary Split) ===
Accuracy : 0.8687323322542164
Precision: 0.23621814318098808
Recall   : 0.9361213455189243
F1 Score : 0.37724370485172926


In [10]:
from sklearn.model_selection import GroupShuffleSplit

# ==========================
# AFTER: Honest grouped split
# ==========================

groups = model_df["content_hash_id"]

X = model_df.drop(columns=["target", "content_hash_id", "ctr"])
y = model_df["target"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

rf = RandomForestClassifier(
    n_estimators=30,
    max_depth=10,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print("=== AFTER (Grouped Split) ===")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

=== AFTER (Grouped Split) ===
Accuracy : 0.8684840595661895
Precision: 0.23653631957614618
Recall   : 0.9353275878487763
F1 Score : 0.37758470727874716


## Before vs After

### Before (ordinary random split)
- Accuracy: 0.8687
- Precision: 0.2362
- Recall: 0.9361
- F1 Score: 0.3772

### After (grouped split)
- Accuracy: 0.8685
- Precision: 0.2365
- Recall: 0.9353
- F1 Score: 0.3776

### Observation

The grouped split produced almost the same performance as the ordinary split. This suggests that the model is not relying on leakage between duplicated or related samples. The evaluation remains stable under a more honest validation strategy, which increases confidence that the reported performance is reliable.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage Audit

| Feature | Leakage Risk | Reason |
|---------|--------------|--------|
| gsc_impressions | Low | Represents search visibility before prediction and is a valid input feature. |
| avg_position | Low | Search ranking information is available before making the prediction. |
| client_has_gsc | None | Client configuration only; does not reveal the target. |
| client_has_ga4 | None | Client configuration only; does not reveal the target. |
| gsc_data_available | None | Indicates data availability and is not derived from the target. |
| ctr | **High** | CTR is calculated using clicks, and the target is based on clicks. This directly leaks information about the target and was removed from the final model. |

### Conclusion

The main leakage risk in the final feature set was the **CTR** feature because it is calculated using `gsc_clicks`, which is also used to create the target label. Including CTR produced unrealistically high performance (100% accuracy), so it was removed before the final evaluation. The remaining features were kept because they do not directly reveal the target.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

"The Random Forest model accurately predicts whether a page will receive clicks."

### Rewritten claim

"In this experiment, the Random Forest model showed promising performance on the available dataset. The results should be interpreted as decision-support rather than proof that the model will generalize to all future data. Performance was measured using both an ordinary split and a grouped split, and the grouped split provides a more honest estimate of model performance."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.